# Post-Training Analysis and Visualization

This notebook is intentionally analysis-only. Run training from Python entrypoints first:

- `uv run python main.py --mode quick`
- `uv run python main.py --mode normal`
- `uv run rl-train --mode normal`

Then use this notebook to load saved artifacts and visualize results.

## 1. Load Post-Training Artifacts

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from analysis.metrics import (
    compute_training_summary,
    format_hall_of_fame,
    load_training_results,
    print_training_diagnostics,
    run_stochastic_audit,
)
from analysis.visualization import (
    plot_attack_success,
    plot_ppo_diagnostics,
    plot_training_metrics,
 )

print("Analysis imports OK")

In [ ]:
OUTPUT_DIR = Path(".")

model_path, metrics, hall_of_fame = load_training_results(str(OUTPUT_DIR))
metrics_path = OUTPUT_DIR / "training_metrics.json"
log_path = OUTPUT_DIR / "logs" / "training_details.log"

print(f"Loaded model path: {model_path}")
print(f"Loaded metrics keys: {len(metrics.keys())}")
print(f"Loaded Hall of Fame entries: {len(hall_of_fame)}")

## 2. Training Metric Visualizations

In [ ]:
fig = plot_training_metrics(str(metrics_path), save_path=str(OUTPUT_DIR / "training_metrics.png"))
plt.show()
print("Saved -> training_metrics.png")

In [ ]:
fig = plot_ppo_diagnostics(str(metrics_path), save_path=str(OUTPUT_DIR / "ppo_diagnostics.png"))
plt.show()
print("Saved -> ppo_diagnostics.png")

## 3. Attack Success and Diagnostics

### Attack Success Chart

In [ ]:
fig = plot_attack_success(str(metrics_path), save_path=str(OUTPUT_DIR / "attack_success.png"))
plt.show()
print("Saved -> attack_success.png")

## 4. Text Diagnostics and Hall of Fame

In [ ]:
print_training_diagnostics(metrics)

summary = compute_training_summary(metrics)
print("\n=== Summary ===")
for key, value in summary.items():
    print(f"{key}: {value}")

In [ ]:
if hall_of_fame:
    display(Markdown(format_hall_of_fame(hall_of_fame)))
else:
    print("No full leaks detected during training.")

## 5. Optional Stochastic Audit

In [ ]:
AUDIT_EPISODES = 5

audit = run_stochastic_audit(
    model_path=model_path,
    base_url="http://127.0.0.1:8000",
    n_episodes=AUDIT_EPISODES,
 )

print("Stochastic audit results:")
print(f"  Episodes: {audit['episodes']}")
print(f"  Success rate: {audit['success_rate']:.2f}")
print(f"  Mean reward: {audit['mean_reward']:.2f}")

## 6. Optional Log Inspection

In [ ]:
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8").splitlines()
    print(f"{log_path} contains {len(lines)} lines")
    print("\n" + "=" * 70)
    print("Last 50 log lines:")
    print("=" * 70)
    print("\n".join(lines[-50:]))
else:
    print(f"No log file found at {log_path}")